In [1]:
from transformers import T5Tokenizer, T5EncoderModel
import torch
import re

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
tokenizer = T5Tokenizer.from_pretrained("Rostlab/ProstT5", do_lower_case=False)
model = T5EncoderModel.from_pretrained("Rostlab/ProstT5", dtype=torch.float16).to(DEVICE)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [4]:
sequence_examples = ["SEQWENCE", "PRTEINO"]
min_length = min([len(s) for s in sequence_examples])
max_length = max([len(s) for s in sequence_examples])
sequence_examples = [" ".join(list(re.sub(r"[UZOB]", "X", seq))) for seq in sequence_examples]
sequence_examples

['S E Q W E N C E', 'P R T E I N X']

In [5]:
sequence_examples = ["<AA2fold>" + " " + seq for seq in sequence_examples]
sequence_examples

['<AA2fold> S E Q W E N C E', '<AA2fold> P R T E I N X']

In [6]:
ids = tokenizer(sequence_examples, add_special_tokens=True, padding="longest", return_tensors="pt").to(DEVICE)
ids

{'input_ids': tensor([[149,   7,   9,  16,  21,   9,  17,  22,   9,   1],
        [149,  13,   8,  11,   9,  12,  17,  23,   1,   0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0]], device='cuda:0')}

In [7]:
with torch.no_grad():
    embeddings = model(ids["input_ids"], attention_mask=ids["attention_mask"])

In [8]:
embeddings.last_hidden_state.shape

torch.Size([2, 10, 1024])

## AA --> 3Di

In [9]:
from transformers import AutoModelForSeq2SeqLM

In [10]:
# translation model
translation_model = AutoModelForSeq2SeqLM.from_pretrained("Rostlab/ProstT5", dtype=torch.float16).to(DEVICE)

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

In [54]:
sequence_examples = ["PRTEINO", "SEQWENCE"]
min_len = min([ len(s) for s in sequence_examples])
max_len = max([ len(s) for s in sequence_examples])

In [55]:
sequence_examples = [" ".join(list(re.sub(r"[UZOB]", "X", sequence))) for sequence in sequence_examples]
sequence_examples

['P R T E I N X', 'S E Q W E N C E']

In [56]:
sequence_examples = [ "<AA2fold>" + " " + s for s in sequence_examples]
sequence_examples

['<AA2fold> P R T E I N X', '<AA2fold> S E Q W E N C E']

In [57]:
ids = tokenizer(sequence_examples, add_special_tokens=True, padding="longest", return_tensors='pt').to(DEVICE)

In [58]:
# config for AA --> 3Di
gen_kwargs_aa2fold = {
                  "do_sample": True,
                  "num_beams": 3,
                  "top_p" : 0.95,
                  "temperature" : 1.2,
                  "top_k" : 6,
                  "repetition_penalty" : 1.2,
}

In [68]:
with torch.no_grad():
    translations = translation_model.generate(
        ids.input_ids,
        attention_mask=ids.attention_mask,
        max_length=max_len,
        min_length=min_len,
        early_stopping=True,
        num_return_sequences=1,
        **gen_kwargs_aa2fold
    )

In [69]:
decoded_translations = tokenizer.batch_decode( translations, skip_special_tokens=True )
structure_sequences = [ "".join(ts.split(" ")) for ts in decoded_translations ] # predicted 3Di strings
structure_sequences

['ddddddd', 'dppppdp']

In [70]:
print(len("PRTEINO"))
print(len(structure_sequences[0]))

7
7


In [72]:
output_length = 0
while output_length != 6:
    with torch.no_grad():
        translations = translation_model.generate(
            ids.input_ids,
            attention_mask=ids.attention_mask,
            max_length=max_len,
            min_length=min_len,
            early_stopping=True,
            num_return_sequences=1,
            **gen_kwargs_aa2fold
        )

    decoded_translations = tokenizer.batch_decode( translations, skip_special_tokens=True )
    structure_sequences = [ "".join(ts.split(" ")) for ts in decoded_translations ] # predicted 3Di strings
    output_length = len(structure_sequences[0])
    print(output_length)

7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
6


In [73]:
ids.attention_mask

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')

y## Inverse: generate AA from the predicted 3Di

In [39]:
sequence_examples_backtranslation = [ "<fold2AA>" + " " + s for s in decoded_translations]
sequence_examples_backtranslation

['<fold2AA> ddddddd', '<fold2AA> dpppppp']

In [40]:
# tokenize sequences and pad up to the longest sequence in the batch
ids_backtranslation = tokenizer(sequence_examples_backtranslation, add_special_tokens=True, padding="longest", return_tensors="pt").to(DEVICE)

In [41]:
gen_kwargs_fold2AA = {
            "do_sample": True,
            "top_p" : 0.90,
            "temperature" : 1.1,
            "top_k" : 6,
            "repetition_penalty" : 1.2,
}

In [45]:
with torch.no_grad():
    backtranslations = translation_model.generate(
        ids_backtranslation.input_ids,
        attention_mask=ids_backtranslation.attention_mask,
        max_length=max_len,
        min_length=min_len,
        early_stopping=True,
        num_return_sequences=1,
        **gen_kwargs_fold2AA
    )

In [47]:
decoded_backtranslations = tokenizer.batch_decode( backtranslations, skip_special_tokens=True )
aminoAcid_sequences = [ "".join(ts.split(" ")) for ts in decoded_backtranslations ] # predicted amino acid strings
aminoAcid_sequences

['VTVELSA', 'MLDTKSE']